In [48]:
import json
import os
from agents import Agent, Runner, handoff
from dotenv import load_dotenv
import requests
import asyncio
import openai
from agents import function_tool
from pydantic import BaseModel
import random
from agents import trace


load_dotenv()

api_key = os.environ.get("OPENAI_API_KEY")


if not api_key:
    raise ValueError(
        "OpenAI API key not found. Please set the OPENAI_API_KEY environment variable."
    )

In [7]:
@function_tool
def get_weather(city: str) -> str:
    """Get the Current Weather for a city"""

    api_key = os.getenv("OPEN_WEATHER_API")

    response = requests.get(
        f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"
    ).json()

    weather = response.get("weather")[0]["description"]
    temp = response["main"]["temp"]
    return f"The current weather in {city} is {weather} with a temperature of {temp}°C."

In [9]:
openai.api_key = api_key

agent = Agent(
    name="Psychology Guide",
    instructions="You are a psychology expert. Provide the most concise explanations and insights on psychological concepts.",
    model="gpt-5-mini",
)


result = await Runner.run(agent, "What is Eisenhower Matrix?")
print(result.final_output)

The Eisenhower Matrix (aka Urgent‑Important Matrix) is a simple decision tool for prioritizing tasks by urgency and importance. It helps shift you from reactive work to focused planning.

Four quadrants:
- Urgent + Important (Do now): crises, deadlines — e.g., finishing a client deliverable due today.
- Not Urgent + Important (Plan): long‑term goals, prevention, growth — e.g., exercise, strategic planning.
- Urgent + Not Important (Delegate): interruptions or other people’s priorities — e.g., routine meeting you can assign.
- Not Urgent + Not Important (Eliminate): time‑wasters — e.g., mindless social media scrolling.

Why it works (psychological basis):
- Reduces decision fatigue by clarifying action steps.
- Encourages self‑regulation and investment in long‑term goals (counteracts present bias).
- Lowers stress by resolving crises and preventing them through planning.

Quick how‑to:
1. List tasks.
2. Put each into a quadrant.
3. Do, schedule, delegate, or delete accordingly.
4. Revie

In [13]:
class cars(BaseModel):
    make: str
    generations: str
    year_released: int
    type: str
    color: str
    still_produced: bool


car_agent = Agent(
    name="Car Agent",
    instructions="You are a car expert. You'll be given a car name and you job is to provide basic information about that car.",
    model="gpt-4o-mini",
    output_type=cars,
)


result1 = await Runner.run(
    car_agent,
    "tell me about the BMW M5 Competition?",
)
print("Structured Output :", result1.final_output, "\n")

data = result1.final_output.model_dump()
print("Python Object : ", data, "\n")

data1 = json.dumps(data, indent=2)
print("json Schema :", data1, "\n")
# schema = result1.final_output.model_json_schema()

Structured Output : make='BMW' generations='F90' year_released=2018 type='Sedan' color='Various (including Black, Silver, Blue, White)' still_produced=True 

Python Object :  {'make': 'BMW', 'generations': 'F90', 'year_released': 2018, 'type': 'Sedan', 'color': 'Various (including Black, Silver, Blue, White)', 'still_produced': True} 

json Schema : {
  "make": "BMW",
  "generations": "F90",
  "year_released": 2018,
  "type": "Sedan",
  "color": "Various (including Black, Silver, Blue, White)",
  "still_produced": true
} 



In [14]:
payload = json.dumps({"data": data}, indent=2)

In [15]:
print(
    f"This is python Object {data} \n and this is json formatted string of same data : {payload}"
)

This is python Object {'make': 'BMW', 'generations': 'F90', 'year_released': 2018, 'type': 'Sedan', 'color': 'Various (including Black, Silver, Blue, White)', 'still_produced': True} 
 and this is json formatted string of same data : {
  "data": {
    "make": "BMW",
    "generations": "F90",
    "year_released": 2018,
    "type": "Sedan",
    "color": "Various (including Black, Silver, Blue, White)",
    "still_produced": true
  }
}


In [18]:
carInfo_agent = Agent(
    name="Car Info Agent",
    instructions="You are a car information expert. You'll be given a Schema about a car and your job is to provide brief information about similar cars that also have those qualities.",
    model="gpt-4o-mini",
)

result = await Runner.run(carInfo_agent, payload)
print(result.final_output)

Here are some similar cars that share qualities with the 2018 BMW F90 generation 5 Series sedan:

1. **Audi A6 (C8)**
   - **Year Released:** 2019
   - **Type:** Sedan
   - **Colors:** Various (including Black, Silver, Blue, White)
   - **Still Produced:** Yes
   - **Highlights:** Known for its luxury, advanced technology, and Quattro all-wheel drive system.

2. **Mercedes-Benz E-Class (W213)**
   - **Year Released:** 2017
   - **Type:** Sedan
   - **Colors:** Various (including Black, Silver, Blue, White)
   - **Still Produced:** Yes
   - **Highlights:** Offers a balance of performance and comfort with a sophisticated interior.

3. **Lexus ES ( seventh generation)**
   - **Year Released:** 2018
   - **Type:** Sedan
   - **Colors:** Various (including Black, Silver, Blue, White)
   - **Still Produced:** Yes
   - **Highlights:** Renowned for its reliability, quiet ride, and luxurious feel.

4. **Genesis G80 (second generation)**
   - **Year Released:** 2017
   - **Type:** Sedan
   - **C

## Using Web Search tool

In [19]:
from agents import WebSearchTool

NewsSearchAgent = Agent(
    name="News Search Agent",
    instructions="You are a news search agent. You'll be given a query and your job is to find the latest news articles related to that query.",
    model="gpt-4o-mini",
    tools=[WebSearchTool()],
)


while True:
    query = input("Enter your Query (or exit using 'quit')")
    if query.lower() == "quit":
        break

    result = await Runner.run(NewsSearchAgent, query)

    print(result.final_output)

## Using Handoffs

In [ ]:
import asyncio
from agents import RunContextWrapper


class Quote(BaseModel):
    personality_name: str
    quote_attributed: str
    language: str


translator_agent_arabic = Agent(
    name="Arabic Translator Agent",
    handoff_description="Translates the given text in Arabic",
    instructions="""You are a translator agent. You'll be given a quote as {"Name": "<the quote of him>"} and your job is to translate the whole text into Arabic language. and give the final output as {"Name": "<the translated quote>"}.""",
    model="gpt-4o-mini",
    output_type=Quote,
)


translator_agent_urdu = Agent(
    name="Urdu Translator Agent",
    handoff_description="Translates the given text in Urdu",
    instructions="""You are a translator agent. You'll be given a quote as {"Name": "<the quote of him>"} and your job is to translate the whole text into Urdu language. and give the final output as {"Name": "<the translated quote>"}.""",
    model="gpt-4o-mini",
    output_type=Quote,
)


def on_urdu_handoff(ctx: RunContextWrapper[None]):
    print("Handing off to Urdu translator agent")


def on_arabic_handoff(ctx: RunContextWrapper[None]):
    print("Handing off to Arabic translator agent")


quote_agent = Agent(
    name="Quote Agent",
    instructions="""
    You are a quote agent. You'll be given a person name and a language name in which it is required in. 
        Find one relevant quote attributed to that name. and hand it to the required translator agent
        if the language is 'Arabic', always hand off to the Arabic translator agent.
        If the language is 'Urdu', always hand off to the Urdu translator agent.
    """,
    model="gpt-5-mini",
    output_type=Quote,
    handoffs=[
        handoff(
            agent=translator_agent_arabic,
            on_handoff=(on_arabic_handoff),
        ),
        handoff(
            agent=translator_agent_urdu,
            on_handoff=(on_urdu_handoff),
        ),
    ],
)
data = []
while True:
    person = input("Enter a personality name to get a quote(or exit using 'quit') ")

    if person.lower() == "quit":
        break
    language = (
        input("Enter the language (arabic/urdu) for translation: ").strip().lower()
    )

    quote = await Runner.run(
        quote_agent, "Give me " + person + "'s quote in " + language + " language"
    )
    print(quote.final_output)
    data.append(quote.final_output.model_dump())
    data.json

Handing off to Arabic translator agent
personality_name='أبو بكر الصديق (رضي الله عنه)' quote_attributed='الصدق منجاة.' language='Arabic'


## Using Handoffs Again

In [51]:
class ManagerEscalation(BaseModel):
    issue: str  # the issue being escalated
    reason: str  # the reason for escalation


@function_tool
def create_ticket(issue: str):
    """Create a ticket for the escalated issue"""
    # Simulate ticket creation
    print(f"Ticket created for issue: {issue}")
    return f"Ticket created successfully ID {random.randint(10000, 99999)}"


manager_agent = Agent(
    name="Manager Agent",
    handoff_description="Handles Escalated Issues that requires managerial oversight",
    instructions="""
    you handle escalated customer issues that the initial customer service agent could not resolve.
    you will recieve the issue and the reason for escalation if the issue could not be resolved for the 
    customer create a ticket and inform the customer.
    """,
    tools=[create_ticket],
    model="gpt-4o-mini",
)


def on_manager_handoff(ctx: RunContextWrapper[None], input: ManagerEscalation):
    print("Escalating to the Manager Agent : \t", input.issue)
    print("Reason For Escalation : \t", input.reason)


customer_service_agent = Agent(
    name="Customer Service Agent",
    handoff_description="Handles initial customer inquiries and issues",
    instructions="""
    you are the first point of contact for customer inquiries.
    if you cannot resolve the issue, escalate it to the manager agent.
    """,
    model="gpt-4o-mini",
    handoffs=[
        handoff(
            manager_agent,
            input_type=ManagerEscalation,
            on_handoff=on_manager_handoff,
        )
    ],
)

with trace("Customer Service Workflow"):
    result = await Runner.run(
        customer_service_agent, "I need a refund but the website is blank"
    )
    print(result.final_output)

Escalating to the Manager Agent : 	 Customer requests a refund but the website is blank, preventing them from completing the process.
Reason For Escalation : 	 Technical issue with the website.
Ticket created for issue: Customer requests a refund but the website is blank, preventing them from completing the process.
I've created a ticket regarding your refund request due to the blank website issue. The ticket ID is **56564**. A manager will follow up on this shortly. Thank you for your patience!


## Tracing

In [ ]:
basic_agent = Agent(
    name="Basic Agent",
    instructions="""
    you are a helpful assitant that responds in the most concise way possible to address user inquiries.
    """,
    model="gpt-4o-mini",
)
with trace("Basic Agent Workflow"):
    result = await Runner.run(
        basic_agent, "who is the owner of the club manchester city and manchester united?"
)

print(result.final_output)

Manchester City is owned by the City Football Group, led by Sheikh Mansour. Manchester United is primarily owned by the Glazer family.
